# Toxicity Fairness Benchmark — Analysis

Pre-computed results from `scripts/run_benchmark.py --sample 1000 --models perspective gemini claude`.

All metrics use 95% bootstrap confidence intervals (1,000 resamples).

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.figure_factory as ff
import plotly.io as pio
from sklearn.metrics import confusion_matrix

pio.renderers.default = 'notebook'

from toxicity_fairness.metrics.fairness import (
    group_stats,
    fairness_report,
    equalized_odds_gap,
    demographic_parity_gap,
    accuracy_gap,
)

## 1. Load results

In [ ]:
df = pd.read_parquet('../results/raw_results.parquet')
print(f'Total rows: {len(df):,}')
print(f'Models: {list(df["model"].unique())}')
print(f'Protected attributes: {sorted(df["protected_attribute"].unique())}')
print(f'\nRows per model:')
print(df['model'].value_counts().to_string())
print(f'\nLabel distribution (actual):')
print(df[df['model'] == df['model'].iloc[0]]['actual_label'].value_counts().to_string())
print(f'\nAttribute distribution:')
print(df[df['model'] == df['model'].iloc[0]]['protected_attribute'].value_counts().to_string())
df.head()

## 2. Per-attribute fairness summary

Accuracy gap = max − min accuracy across subgroups within each attribute. Smaller = more equitable.

In [ ]:
ATTRS = ['Gender', 'Race/Ethnicity', 'Religion']
MIN_N = 20

rows = []
for model, mdf in df.groupby('model'):
    overall_acc = (mdf['actual_label'] == mdf['predicted_label']).mean()
    row = {'model': model.split('/')[-1], 'overall_accuracy': overall_acc}
    for attr in ATTRS:
        adf = mdf[mdf['protected_attribute'] == attr]
        col = attr.split('/')[0].lower() + '_acc_gap'
        if len(adf) >= MIN_N:
            row[col] = accuracy_gap(group_stats(adf))
        else:
            row[col] = float('nan')
    rows.append(row)

summary = pd.DataFrame(rows).set_index('model')
gap_cols = [c for c in summary.columns if 'gap' in c]
summary.style.format('{:.1%}', na_rep='n/a') \
    .background_gradient(subset=gap_cols, cmap='RdYlGn_r')

## 3. Accuracy by subgroup

Bar charts with 95% bootstrap confidence intervals. Smaller gaps between bars within each model = more equitable. Hover for sample size.

In [ ]:
def accuracy_bar_chart(df, attribute, title):
    adf = df[df['protected_attribute'] == attribute]
    rows = []
    for model, mdf in adf.groupby('model'):
        stats = group_stats(mdf)
        for grp, row in stats.iterrows():
            if row['n'] >= 10:
                rows.append({
                    'model': model.split('/')[-1], 'group': str(grp),
                    'accuracy': row['accuracy'],
                    'err_hi': row['acc_ci_hi'] - row['accuracy'],
                    'err_lo': row['accuracy'] - row['acc_ci_lo'],
                    'n': row['n'],
                })
    if not rows:
        print(f'Not enough data for {attribute}')
        return
    bar_df = pd.DataFrame(rows)
    fig = px.bar(
        bar_df, x='group', y='accuracy', color='model', barmode='group',
        error_y='err_hi', error_y_minus='err_lo',
        title=title,
        color_discrete_sequence=px.colors.qualitative.Set2,
        hover_data=['n'],
        labels={'accuracy': 'Accuracy', 'group': attribute, 'model': 'Model'},
    )
    fig.update_layout(yaxis_tickformat='.0%', yaxis_range=[0, 1])
    fig.show()

In [ ]:
accuracy_bar_chart(df, 'Gender', 'Accuracy by gender subgroup (95% bootstrap CI)')

In [ ]:
accuracy_bar_chart(df, 'Race/Ethnicity', 'Accuracy by race/ethnicity subgroup (95% bootstrap CI)')

In [ ]:
accuracy_bar_chart(df, 'Religion', 'Accuracy by religion subgroup (95% bootstrap CI)')

## 4. Equalized odds: TPR vs. FPR

A model satisfies equalized odds if it has equal TPR (sensitivity) and FPR (false alarm rate) across groups. Points clustered together = more equitable. The diagonal is the random-classifier baseline.

In [ ]:
def equalized_odds_scatter(df, attribute, title):
    adf = df[df['protected_attribute'] == attribute]
    rows = []
    for model, mdf in adf.groupby('model'):
        stats = group_stats(mdf)
        for grp, row in stats.iterrows():
            if row['n'] >= 10:
                rows.append({
                    'model': model.split('/')[-1], 'group': str(grp),
                    'tpr': row['tpr'], 'fpr': row['fpr'], 'n': row['n'],
                })
    if not rows:
        print(f'Not enough data for {attribute}')
        return
    eo_df = pd.DataFrame(rows)
    fig = px.scatter(
        eo_df, x='fpr', y='tpr', color='model', text='group', size='n',
        title=title,
        color_discrete_sequence=px.colors.qualitative.Set2,
        labels={'fpr': 'False Positive Rate', 'tpr': 'True Positive Rate'},
    )
    fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1,
                  line=dict(color='gray', dash='dash'))
    fig.update_traces(textposition='top center')
    fig.update_layout(xaxis_range=[-0.05, 1.05], yaxis_range=[-0.05, 1.05])
    fig.show()

In [ ]:
equalized_odds_scatter(df, 'Gender', 'Equalized odds by gender subgroup')

In [ ]:
equalized_odds_scatter(df, 'Race/Ethnicity', 'Equalized odds by race/ethnicity subgroup')

## 5. Confusion matrices

Per-model confusion matrices on the full 1,000-sample set (de-duplicated by text).

In [ ]:
LABELS = ['toxic', 'non-toxic']

for model in sorted(df['model'].unique()):
    mdf = df[df['model'] == model].drop_duplicates(subset=['text'])
    cm = confusion_matrix(mdf['actual_label'], mdf['predicted_label'], labels=LABELS)
    pct = cm.astype(float) / cm.sum()
    annotations = [[f'{cm[i][j]}<br>({pct[i][j]:.0%})' for j in range(2)] for i in range(2)]
    fig = ff.create_annotated_heatmap(
        cm,
        x=[f'pred: {l}' for l in LABELS],
        y=[f'actual: {l}' for l in LABELS],
        annotation_text=annotations,
        colorscale='Blues',
    )
    fig.update_layout(
        title=f'Confusion matrix — {model.split("/")[-1]}',
        xaxis_title='Predicted',
        yaxis_title='Actual',
    )
    fig.show()

## 6. Numbers for README key findings table

Copy the printed table below into `README.md`.

In [ ]:
ATTR_COLS = [('Gender', 'Gender Gap'), ('Race/Ethnicity', 'Race Gap'), ('Religion', 'Religion Gap')]
header_row = '| Model | Overall Accuracy | ' + ' | '.join(h for _, h in ATTR_COLS) + ' |'
sep_row = '|---|' + '---|' * (1 + len(ATTR_COLS))

print('=== README key findings table ===')
print()
print(header_row)
print(sep_row)

for model in sorted(df['model'].unique()):
    mdf = df[df['model'] == model]
    overall = (mdf['actual_label'] == mdf['predicted_label']).mean()
    label = model.split('/')[-1]
    gaps = []
    for attr, _ in ATTR_COLS:
        adf = mdf[mdf['protected_attribute'] == attr]
        if len(adf) >= MIN_N:
            gap_pp = accuracy_gap(group_stats(adf)) * 100
            gaps.append(f'{gap_pp:.0f} pp')
        else:
            gaps.append('n/a')
    print(f'| {label} | {overall:.0%} | ' + ' | '.join(gaps) + ' |')

## 7. Key findings

In [ ]:
accs = {}
gender_gaps = {}
race_gaps = {}

for model, mdf in df.groupby('model'):
    short = model.split('/')[-1]
    accs[short] = (mdf['actual_label'] == mdf['predicted_label']).mean()
    for attr, d in [('Gender', gender_gaps), ('Race/Ethnicity', race_gaps)]:
        adf = mdf[mdf['protected_attribute'] == attr]
        if len(adf) >= MIN_N:
            d[short] = accuracy_gap(group_stats(adf))

best_acc = max(accs, key=accs.get)
worst_acc = min(accs, key=accs.get)
most_fair_gender = min(gender_gaps, key=gender_gaps.get) if gender_gaps else None

print('ACCURACY RANKING')
print('-' * 40)
for model, acc in sorted(accs.items(), key=lambda x: -x[1]):
    bar = '\u2588' * int(acc * 20)
    print(f'  {model:30s} {acc:.1%}  {bar}')

if gender_gaps:
    print()
    print('GENDER ACCURACY GAP (smaller = fairer)')
    print('-' * 40)
    for model, gap in sorted(gender_gaps.items(), key=lambda x: x[1]):
        bar = '\u2588' * int(gap * 100)
        print(f'  {model:30s} {gap*100:.1f} pp  {bar}')

if race_gaps:
    print()
    print('RACE/ETHNICITY ACCURACY GAP (smaller = fairer)')
    print('-' * 40)
    for model, gap in sorted(race_gaps.items(), key=lambda x: x[1]):
        bar = '\u2588' * int(gap * 100)
        print(f'  {model:30s} {gap*100:.1f} pp  {bar}')

print()
print(f'Most accurate model:         {best_acc} ({accs[best_acc]:.1%})')
if most_fair_gender:
    print(f'Most gender-fair model:      {most_fair_gender} ({gender_gaps[most_fair_gender]*100:.1f} pp gap)')
    if best_acc == most_fair_gender:
        print(f'\n=> {best_acc} achieves both best accuracy AND smallest gender fairness gap.')
    else:
        print(f'\n=> Accuracy-fairness tradeoff: {best_acc} is most accurate; {most_fair_gender} is most gender-fair.')

## 8. Limitations

- **Sample size**: 1,000 rows split across 3 models and multiple subgroups. Some subgroups may have <20 samples, limiting statistical power.
- **Protected attribute proxies**: HateXplain's target labels (e.g., "women", "African") are annotator-assigned proxies, not ground truth demographic identifiers.
- **Prompt sensitivity**: Gemini and Claude are LLM-based classifiers whose output depends on prompt wording. Results reflect the prompts in `docs/prompt_design.md`.
- **API drift**: Results reflect API behavior at time of evaluation. Model updates may change scores.
- **Dataset domain**: HateXplain is a social media hate speech dataset. Performance may not generalize to other text domains.